In [ ]:
import os
import sys
import gc
import time
import csv
import math
import warnings
import pickle
import numpy as np
import pandas as pd
import joblib
import rasterio
from rasterio.windows import Window
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed
import multiprocessing as mp
import cupy as cp
import cupyx.scipy.ndimage as cpnd
import cuml
from cuml.fil import ForestInference
from tqdm import tqdm

warnings.filterwarnings('ignore')

RUN_FOR = 'SOC0_200'

B              = 50
N_PREDICT      = 10
N_PREDICT_SEED = 0
RANDOM_SEED    = 42
N_JOBS_TRAIN   = -1

GPU_INFERENCE  = 0
GPU_IDS        = [0, 1]

FIL_BATCH_SIZE = 100_000
BATCH_B        = 10

TILE_SIZE  = 3840
NODATA_OUT = -9999.0

SOC_UPPER_LIMIT = 2000

PIXEL_SIZE_M     = 10
INPAINT_RADIUS_M = 3_000
INPAINT_SIGMA_M  =   800

INPAINT_RADIUS_PX  = max(1, int(round(INPAINT_RADIUS_M / PIXEL_SIZE_M)))
INPAINT_SIGMA_PX   = max(1, int(round(INPAINT_SIGMA_M  / PIXEL_SIZE_M)))
INPAINT_MAX_ITER   = 8

INPAINT_SKIP_RATIO = 0.001

INPAINT_WEIGHT_EPS = cp.float32(1e-4)

ANCHOR_BANDS = {
    'B2', 'B5', 'B6',
    'GRVI', 'MVI', 'NBR',
    'MNDWI',
    'Clre',
}

BASE1    = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff'
BASE2    = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-2'
BASE3    = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-3'
BASE4    = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-4'
BASE5    = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-5'
BASE6    = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-6'
SHP_PATH = '/kaggle/input/datasets/antsasarobidyran/mangove-data-v1'

RASTERS = {
    'BIO1_MAT':   (f'{BASE1}/BIO1.tif',        True,  10.0),
    'BIO12_MAP':  (f'{BASE1}/BIO12.tif',        True,  None),
    'CHM':        (f'{BASE1}/CHM.tif',          False, None),
    'SST_DRY':    (f'{BASE1}/SST_DRY.tif',      False, None),
    'TREECOVER':  (f'{BASE1}/TREECOVER.tif',    False, None),
    'Clre':       (f'{BASE2}/Clre.tif',         False, None),
    'NBR':        (f'{BASE2}/NBR.tif',          False, None),
    'SLOPE':      (f'{BASE2}/SLOPE.tif',        False, None),
    'SST_WET':    (f'{BASE2}/SST_WET.tif',      False, None),
    'TWI':        (f'{BASE5}/TWI.tif',          False, None),
    'B6':         (f'{BASE3}/B6.tif',           False, None),
    'GRVI':       (f'{BASE3}/GRVI.tif',         False, None),
    'MNDWI':      (f'{BASE3}/MNDWI.tif',        False, None),
    'MVI':        (f'{BASE3}/MVI.tif',          False, None),
    'DIST_WATER': (f'{BASE4}/DIST_WATER.tif',   False, None),
    'B2':         (f'{BASE6}/B2.tif',           False, None),
    'B5':         (f'{BASE6}/B5.tif',           False, None),
}

VIF_DIR = '/kaggle/working/VIF'

RESPONSE_CONFIG = {
    'AGC': {
        'train_csv':  '/kaggle/working/Mangrove_AGC_clean_full.csv',
        'model_pkl':  '/kaggle/working/ML_Results/AGC/rf_model_AGC.pkl',
        'vif_csv':    os.path.join(VIF_DIR, 'vif_final_AGC.csv'),
        'response':   'AGC',
        'output_dir': '/kaggle/working/Maps/AGC',
    },
    'BGC': {
        'train_csv':  '/kaggle/working/Mangrove_BGC_clean_full.csv',
        'model_pkl':  '/kaggle/working/ML_Results/BGC/rf_model_BGC.pkl',
        'vif_csv':    os.path.join(VIF_DIR, 'vif_final_BGC.csv'),
        'response':   'BGC',
        'output_dir': '/kaggle/working/Maps/BGC',
    },
    'SOC0_30': {
        'train_csv':  '/kaggle/working/Mangrove_SOC0_30_clean_full.csv',
        'model_pkl':  '/kaggle/working/ML_Results/SOC0_30/rf_model_SOC0_30.pkl',
        'vif_csv':    os.path.join(VIF_DIR, 'vif_final_SOC0_30.csv'),
        'response':   'SOC 0- 30',
        'output_dir': '/kaggle/working/Maps/SOC0_30',
    },
    'SOC0_100': {
        'train_csv':  '/kaggle/working/Mangrove_SOC0_100_clean_full.csv',
        'model_pkl':  '/kaggle/working/ML_Results/SOC0_100/rf_model_SOC0_100.pkl',
        'vif_csv':    os.path.join(VIF_DIR, 'vif_final_SOC0_100.csv'),
        'response':   'SOC 0- 100',
        'output_dir': '/kaggle/working/Maps/SOC0_100',
    },
    'SOC0_200': {
        'train_csv':  '/kaggle/working/Mangrove_SOC0_200_clean_full.csv',
        'model_pkl':  '/kaggle/working/ML_Results/SOC0_200/rf_model_SOC0_200.pkl',
        'vif_csv':    os.path.join(VIF_DIR, 'vif_final_SOC0_200.csv'),
        'response':   'SOC 0- 200',
        'output_dir': '/kaggle/working/Maps/SOC0_200',
    },
    'TECS': {
        'train_csv':  '/kaggle/working/Mangrove_TECS_clean_full.csv',
        'model_pkl':  '/kaggle/working/ML_Results/TECS/rf_model_TECS.pkl',
        'vif_csv':    os.path.join(VIF_DIR, 'vif_final_TECS.csv'),
        'response':   'TECS',
        'output_dir': '/kaggle/working/Maps/TECS',
    },
}

assert RUN_FOR in RESPONSE_CONFIG, f"RUN_FOR must be one of {list(RESPONSE_CONFIG.keys())}"
cfg           = RESPONSE_CONFIG[RUN_FOR]
TRAIN_CSV     = cfg['train_csv']
MODEL_PKL     = cfg['model_pkl']
VIF_CSV       = cfg['vif_csv']
RESPONSE      = cfg['response']
OUTPUT_DIR    = cfg['output_dir']
BOOTSTRAP_PKL = os.path.join(OUTPUT_DIR, f'bootstrap_models_{RUN_FOR}.pkl')

os.makedirs(OUTPUT_DIR, exist_ok=True)

_n_gpus = cp.cuda.runtime.getDeviceCount()
if _n_gpus < 2:
    GPU_IDS = [0]
GPU_INFERENCE = GPU_IDS[0]
cp.cuda.Device(GPU_INFERENCE).use()

def raster_path(name):           return RASTERS[name][0]
def raster_zero_is_nodata(name): return RASTERS[name][1]
def raster_scale(name):          return RASTERS[name][2]


def validate_raster_paths(active_names):
    missing = [f'  {n}: {raster_path(n)}'
               for n in active_names if not os.path.exists(raster_path(n))]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} raster(s) not found:\n" + "\n".join(missing))


def load_features(vif_csv):
    if not os.path.exists(vif_csv):
        raise FileNotFoundError(
            f"VIF CSV not found: {vif_csv}\n"
            f"Available in VIF_DIR: {os.listdir(VIF_DIR)}")
    features = pd.read_csv(vif_csv)['predictor'].tolist()
    missing = [f for f in features if f not in RASTERS]
    if missing:
        raise ValueError(f"Features in VIF CSV but no raster defined: {missing}")
    return features


def compute_training_medians(features, train_csv):
    df = pd.read_csv(train_csv).replace(-9999, np.nan).dropna(subset=[RESPONSE])
    if SOC_UPPER_LIMIT and RUN_FOR.startswith('SOC'):
        df = df[df[RESPONSE] < SOC_UPPER_LIMIT].copy()
    medians = {}
    for col in features:
        scale = raster_scale(col)
        vals  = df[col].dropna() if col in df.columns else pd.Series([], dtype=float)
        if scale is not None:
            vals = vals / scale
        medians[col] = float(vals.median()) if len(vals) > 0 else 0.0
    return medians


def _inpaint_one_band_on_device(arr_cpu: np.ndarray,
                                 mask_cpu: np.ndarray,
                                 fallback: float,
                                 sigma_px: int,
                                 max_iter: int,
                                 device_id: int) -> np.ndarray:
    n_pixels  = arr_cpu.size
    n_missing = int(mask_cpu.sum())

    if n_missing == 0:
        return arr_cpu
    if n_missing / n_pixels < INPAINT_SKIP_RATIO:
        out = arr_cpu.copy()
        out[mask_cpu] = fallback
        return out

    with cp.cuda.Device(device_id):
        filled  = cp.asarray(arr_cpu,  dtype=cp.float32)
        to_fill = cp.asarray(mask_cpu, dtype=cp.bool_)
        fb      = cp.float32(fallback)

        valid_f = (~to_fill).astype(cp.float32)

        for _ in range(max_iter):
            if not to_fill.any():
                break

            data_f = filled.copy()
            data_f[to_fill] = cp.float32(0.0)

            bd = cpnd.gaussian_filter(
                data_f.astype(cp.float32), sigma=sigma_px, mode='nearest'
            )
            bw = cpnd.gaussian_filter(
                valid_f.astype(cp.float32), sigma=sigma_px, mode='nearest'
            )
            del data_f

            can   = to_fill & (bw > INPAINT_WEIGHT_EPS)
            n_can = int(can.sum())
            if n_can == 0:
                del bd, bw, can
                break

            filled[can]  = bd[can] / bw[can]
            to_fill[can] = False

            valid_f[can] = cp.float32(1.0)

            del bd, bw, can

        if to_fill.any():
            filled[to_fill] = fb

        result = cp.asnumpy(filled)
        del filled, to_fill, valid_f
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()

    return result


def inpaint_optional_bands_multigpu(bands: dict,
                                    band_missing: dict,
                                    optional_names: list,
                                    train_medians: dict,
                                    sigma_px: int,
                                    max_iter: int,
                                    gpu_ids: list) -> dict:
    work = []
    for name in optional_names:
        msk = band_missing.get(name)
        if msk is not None and msk.any():
            work.append(name)

    if not work:
        return bands

    n_gpus = len(gpu_ids)

    if n_gpus == 1:
        for name in work:
            bands[name] = _inpaint_one_band_on_device(
                bands[name], band_missing[name],
                train_medians[name], sigma_px, max_iter, gpu_ids[0],
            )
        return bands

    from concurrent.futures import ThreadPoolExecutor, as_completed

    futures = {}
    with ThreadPoolExecutor(max_workers=min(len(work), n_gpus)) as pool:
        for i, name in enumerate(work):
            dev = gpu_ids[i % n_gpus]
            fut = pool.submit(
                _inpaint_one_band_on_device,
                bands[name],
                band_missing[name],
                train_medians[name],
                sigma_px,
                max_iter,
                dev,
            )
            futures[fut] = name

        for fut in as_completed(futures):
            name         = futures[fut]
            bands[name]  = fut.result()

    return bands


FEATURES = load_features(VIF_CSV)

active_anchors  = [f for f in FEATURES if f in ANCHOR_BANDS]
active_optional = [f for f in FEATURES if f not in ANCHOR_BANDS]

validate_raster_paths(FEATURES)

TRAIN_MEDIANS = compute_training_medians(FEATURES, TRAIN_CSV)


def _train_one(b, X_full, y_full, rf_params, n_samples, seed):
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import StandardScaler
    import numpy as np, pickle

    rng      = np.random.RandomState(seed + b)
    boot_idx = rng.randint(0, n_samples, size=n_samples)

    oob_mask                      = np.ones(n_samples, dtype=bool)
    oob_mask[np.unique(boot_idx)] = False
    oob_idx                       = np.where(oob_mask)[0]

    X_boot = X_full[boot_idx];  y_boot = y_full[boot_idx]
    scaler = StandardScaler();  X_sc   = scaler.fit_transform(X_boot)

    params = {**rf_params, 'random_state': seed + b, 'n_jobs': 1}
    mdl    = RandomForestRegressor(**params)
    mdl.fit(X_sc, y_boot)

    oob_r2 = oob_rmse = np.nan
    if len(oob_idx) > 1:
        p_oob    = mdl.predict(scaler.transform(X_full[oob_idx]))
        y_oob    = y_full[oob_idx]
        ss_res   = float(np.sum((y_oob - p_oob) ** 2))
        ss_tot   = float(np.sum((y_oob - y_oob.mean()) ** 2))
        oob_r2   = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        oob_rmse = float(np.sqrt(np.mean((y_oob - p_oob) ** 2)))

    return dict(
        model_bytes = pickle.dumps(mdl),
        sc_mean     = scaler.mean_.astype(np.float32),
        sc_scale    = scaler.scale_.astype(np.float32),
        oob_r2      = float(oob_r2),
        oob_rmse    = float(oob_rmse),
    )


def phase1_train(features, rf_params):
    df = pd.read_csv(TRAIN_CSV).replace(-9999, np.nan).dropna(subset=[RESPONSE])
    if SOC_UPPER_LIMIT and RUN_FOR.startswith('SOC'):
        df = df[df[RESPONSE] < SOC_UPPER_LIMIT].copy()

    for col in features:
        if col in df.columns and df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    for col in features:
        scale = raster_scale(col)
        if scale is not None and col in df.columns:
            df[col] = df[col] / scale

    X_full    = df[features].values.astype(np.float32)
    y_full    = df[RESPONSE].values.astype(np.float32)
    n_samples = len(y_full)

    t0 = time.time()

    results = Parallel(n_jobs=N_JOBS_TRAIN, verbose=5, backend='loky')(
        delayed(_train_one)(b, X_full, y_full, rf_params, n_samples, RANDOM_SEED)
        for b in range(B)
    )

    oob_r2s  = np.array([r['oob_r2']   for r in results])
    oob_rmse = np.array([r['oob_rmse'] for r in results])

    oob_csv = os.path.join(OUTPUT_DIR, f'bootstrap_oob_{RUN_FOR}.csv')
    with open(oob_csv, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['replicate', 'oob_r2', 'oob_rmse_MgCha'])
        for b in range(B):
            w.writerow([b, f'{oob_r2s[b]:.6f}', f'{oob_rmse[b]:.6f}'])

    joblib.dump({'models': results, 'features': features}, BOOTSTRAP_PKL, compress=3)
    return results


def _select_predict_indices(B_eff):
    if N_PREDICT is None or N_PREDICT >= B_eff:
        indices = list(range(B_eff))
    else:
        rng     = np.random.RandomState(N_PREDICT_SEED)
        indices = sorted(rng.choice(B_eff, size=N_PREDICT, replace=False).tolist())
    return indices


def _fil_predict(fil_model, X_gpu):
    raw = fil_model.predict(X_gpu)
    if hasattr(raw, 'values'):
        raw = raw.values
    if isinstance(raw, np.ndarray):
        raw = cp.asarray(raw)
    return raw.ravel().astype(cp.float32)


def phase2_predict(features, models):
    B_eff           = len(models)
    predict_indices = _select_predict_indices(B_eff)
    N_eff           = len(predict_indices)
    n_batches       = math.ceil(N_eff / BATCH_B)

    idx_csv = os.path.join(OUTPUT_DIR, f'predict_model_indices_{RUN_FOR}.csv')
    with open(idx_csv, 'w', newline='') as f:
        csv.writer(f).writerow(['selected_bootstrap_index'] + predict_indices)

    src_nodata = {}
    with rasterio.open(raster_path(features[0])) as ref:
        profile = ref.profile.copy()
        H, W    = ref.height, ref.width
        crs     = ref.crs
    for name in features:
        with rasterio.open(raster_path(name)) as src:
            src_nodata[name] = src.nodata

    row_starts = list(range(0, H, TILE_SIZE))
    col_starts = list(range(0, W, TILE_SIZE))
    n_tiles    = len(row_starts) * len(col_starts)
    n_cols_t   = len(col_starts)
    tiles = [
        (ti * n_cols_t + tj + 1, row_starts[ti], col_starts[tj])
        for ti in range(len(row_starts))
        for tj in range(len(col_starts))
    ]

    raster_handles = {name: rasterio.open(raster_path(name)) for name in features}

    def _build_X_cpu(rs, cs):
        re  = min(rs + TILE_SIZE, H);  ce = min(cs + TILE_SIZE, W)
        th  = re - rs;                 tw = ce - cs
        win = Window(cs, rs, tw, th)

        bands        = {}
        valid_anchor = np.ones((th, tw), dtype=bool)
        band_missing = {}

        for name, src in raster_handles.items():
            arr = src.read(1, window=win).astype(np.float32)
            if arr.shape != (th, tw):
                tmp = np.full((th, tw), np.nan, np.float32)
                tmp[:arr.shape[0], :arr.shape[1]] = arr
                arr = tmp

            nd  = src_nodata[name]
            msk = ~np.isfinite(arr)
            if nd is not None:
                msk |= np.isclose(arr, nd)
            if raster_zero_is_nodata(name):
                msk |= (arr == 0)

            if name in ANCHOR_BANDS:
                valid_anchor &= ~msk
            else:
                band_missing[name] = msk

            bands[name] = arr

        n_v = int(valid_anchor.sum())
        if n_v == 0:
            return None, None, th, tw, win

        bands = inpaint_optional_bands_multigpu(
            bands          = bands,
            band_missing   = band_missing,
            optional_names = active_optional,
            train_medians  = TRAIN_MEDIANS,
            sigma_px       = INPAINT_SIGMA_PX,
            max_iter       = INPAINT_MAX_ITER,
            gpu_ids        = GPU_IDS,
        )

        flat = np.where(valid_anchor.ravel())[0].astype(np.int32)
        cols = []
        for f in features:
            col   = bands[f].ravel()[flat].copy()
            scale = raster_scale(f)
            if scale is not None:
                col /= scale
            cols.append(col)

        bands.clear()
        band_missing.clear()
        return np.column_stack(cols).astype(np.float32), flat, th, tw, win

    def _gpu_predict(fil, X_cpu, sc_mean, sc_scale):
        with cp.cuda.Device(GPU_INFERENCE):
            X_sc = (X_cpu - sc_mean) / sc_scale
            n_v  = X_cpu.shape[0]
            out  = np.empty(n_v, dtype=np.float32)
            for p0 in range(0, n_v, FIL_BATCH_SIZE):
                p1         = min(p0 + FIL_BATCH_SIZE, n_v)
                X_gpu      = cp.asarray(X_sc[p0:p1])
                out[p0:p1] = cp.asnumpy(_fil_predict(fil, X_gpu))
                del X_gpu
                cp.get_default_memory_pool().free_all_blocks()
            del X_sc
        return out

    tile_accums: dict = {}
    tile_meta:   dict = {}

    tracking_csv    = os.path.join(OUTPUT_DIR, f'tile_tracking_{RUN_FOR}.csv')
    tracking_fields = [
        'batch_idx', 'batch_global_models',
        'tile_idx', 'row_start', 'col_start',
        'n_valid_pixels', 'models_accumulated',
        'inpaint_s', 'batch_elapsed_s', 'cumulative_elapsed_min', 'status',
    ]
    tracking_file   = open(tracking_csv, 'w', newline='')
    tracking_writer = csv.DictWriter(tracking_file, fieldnames=tracking_fields)
    tracking_writer.writeheader()
    tracking_file.flush()

    out_prof = profile.copy()
    out_prof.update(
        dtype='float32', count=1, nodata=NODATA_OUT,
        compress='lzw', predictor=2,
        tiled=True, blockxsize=512, blockysize=512, bigtiff='YES',
    )
    PATH_MEAN = os.path.join(OUTPUT_DIR, f'{RUN_FOR}_mean_n{N_eff}.tif')
    PATH_STD  = os.path.join(OUTPUT_DIR, f'{RUN_FOR}_std_n{N_eff}.tif')

    t0 = time.time()

    for batch_idx in range(n_batches):
        local_start   = batch_idx * BATCH_B
        local_end     = min(local_start + BATCH_B, N_eff)
        batch_locals  = list(range(local_start, local_end))
        batch_globals = [predict_indices[li] for li in batch_locals]

        t_c = time.time()

        with cp.cuda.Device(GPU_INFERENCE):
            fil_batch, mean_batch, scale_batch = [], [], []
            for g_idx in batch_globals:
                sk = pickle.loads(models[g_idx]['model_bytes'])
                fil_batch.append(ForestInference.load_from_sklearn(sk))
                mean_batch.append(models[g_idx]['sc_mean'].astype(np.float32))
                scale_batch.append(models[g_idx]['sc_scale'].astype(np.float32))
                del sk

        pbar = tqdm(
            tiles,
            desc=f'Batch {batch_idx+1}/{n_batches}',
            unit='tile', ncols=95,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        )

        for tidx, rs, cs in pbar:
            t_tile = time.time()
            t_inp  = 0.0

            try:
                t_inp_start              = time.time()
                X_cpu, flat, th, tw, win = _build_X_cpu(rs, cs)
                t_inp                    = time.time() - t_inp_start
                read_ok                  = True
            except Exception as e:
                X_cpu   = None
                read_ok = False

            if X_cpu is None:
                tracking_writer.writerow(dict(
                    batch_idx              = batch_idx,
                    batch_global_models    = str(batch_globals),
                    tile_idx               = tidx,
                    row_start              = rs,
                    col_start              = cs,
                    n_valid_pixels         = 0,
                    models_accumulated     = tile_accums[tidx]['k']
                                             if tidx in tile_accums else 0,
                    inpaint_s              = f'{t_inp:.2f}',
                    batch_elapsed_s        = f'{time.time()-t_tile:.2f}',
                    cumulative_elapsed_min = f'{(time.time()-t0)/60:.2f}',
                    status                 = 'nodata' if read_ok else 'error',
                ))
                tracking_file.flush()
                continue

            n_v = X_cpu.shape[0]

            if tidx not in tile_accums:
                tile_accums[tidx] = {
                    'mean': np.zeros(n_v, np.float32),
                    'm2':   np.zeros(n_v, np.float32),
                    'k':    0,
                }
                tile_meta[tidx] = {'flat': flat, 'th': th, 'tw': tw, 'win': win}

            mean_v = tile_accums[tidx]['mean']
            m2_v   = tile_accums[tidx]['m2']
            k      = tile_accums[tidx]['k']

            for j in range(len(batch_globals)):
                try:
                    preds = _gpu_predict(fil_batch[j], X_cpu,
                                         mean_batch[j], scale_batch[j])
                except Exception as e:
                    continue

                k     += 1
                delta  = preds - mean_v
                mean_v = mean_v + delta / np.float32(k)
                m2_v   = m2_v   + delta * (preds - mean_v)
                del preds

            tile_accums[tidx]['mean'] = mean_v
            tile_accums[tidx]['m2']   = m2_v
            tile_accums[tidx]['k']    = k

            tracking_writer.writerow(dict(
                batch_idx              = batch_idx,
                batch_global_models    = str(batch_globals),
                tile_idx               = tidx,
                row_start              = rs,
                col_start              = cs,
                n_valid_pixels         = n_v,
                models_accumulated     = k,
                inpaint_s              = f'{t_inp:.2f}',
                batch_elapsed_s        = f'{time.time()-t_tile:.2f}',
                cumulative_elapsed_min = f'{(time.time()-t0)/60:.2f}',
                status                 = 'ok',
            ))
            tracking_file.flush()
            del X_cpu
            gc.collect()

        pbar.close()

        with cp.cuda.Device(GPU_INFERENCE):
            for fil in fil_batch:
                del fil
            del fil_batch, mean_batch, scale_batch
            cp.get_default_memory_pool().free_all_blocks()
        gc.collect()

        elapsed = time.time() - t0
        eta     = elapsed / (batch_idx + 1) * (n_batches - batch_idx - 1)

    tracking_file.close()

    for h in raster_handles.values():
        h.close()
    for dev in GPU_IDS:
        with cp.cuda.Device(dev):
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()

    with rasterio.open(PATH_MEAN, 'w', **out_prof) as h_mean, \
         rasterio.open(PATH_STD,  'w', **out_prof) as h_std:

        for tidx, rs, cs in tqdm(tiles, desc='Writing tiles',
                                  unit='tile', ncols=80):
            re  = min(rs + TILE_SIZE, H);  ce = min(cs + TILE_SIZE, W)
            th  = re - rs;                 tw = ce - cs
            win = Window(cs, rs, tw, th)

            mt = np.full((th, tw), NODATA_OUT, np.float32)
            st = np.full((th, tw), NODATA_OUT, np.float32)

            if tidx in tile_accums:
                flat       = tile_meta[tidx]['flat']
                r2, c2     = np.unravel_index(flat, (th, tw))
                k_used     = tile_accums[tidx]['k']
                variance   = tile_accums[tidx]['m2'] / max(k_used - 1, 1)
                mt[r2, c2] = tile_accums[tidx]['mean']
                st[r2, c2] = np.sqrt(np.maximum(variance, 0.0))

            h_mean.write(mt, 1, window=win)
            h_std .write(st, 1, window=win)

    tile_accums.clear()
    tile_meta.clear()
    for dev in GPU_IDS:
        with cp.cuda.Device(dev):
            cp.get_default_memory_pool().free_all_blocks()

    elapsed = time.time() - t0


def main():
    saved     = joblib.load(cfg['model_pkl'])
    model     = saved['model']
    rf_params = model.get_params()
    for k in ('n_jobs', 'random_state', 'oob_score', 'warm_start'):
        rf_params.pop(k, None)

    if os.path.exists(BOOTSTRAP_PKL):
        models = joblib.load(BOOTSTRAP_PKL)['models']
    else:
        models = phase1_train(FEATURES, rf_params)

    phase2_predict(FEATURES, models)

    N_eff = N_PREDICT if (N_PREDICT is not None and N_PREDICT < B) else B
    for fname in (
        f'{RUN_FOR}_mean_n{N_eff}.tif',
        f'{RUN_FOR}_std_n{N_eff}.tif',
        f'bootstrap_oob_{RUN_FOR}.csv',
        f'bootstrap_models_{RUN_FOR}.pkl',
        f'predict_model_indices_{RUN_FOR}.csv',
        f'tile_tracking_{RUN_FOR}.csv',
    ):
        p = os.path.join(OUTPUT_DIR, fname)


if __name__ == '__main__':
    main()